# Module 2 Chapter 6：对抗样本检测（Adversarial Sample Detection）

本 notebook 介绍**对抗样本检测**技术：训练一个二分类器来区分干净输入与对抗样本。检测可视为异常检测或二分类问题，常见方法包括：
- 使用副分类器（secondary classifier）在原始图像或模型输出上区分对抗样本；
- 基于 PCA 的统计异常检测；
- 基于预测不一致性的检测。

检测是对抗防御体系中的重要一环，但单独使用时难以应对自适应攻击（adaptive attacks）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# SimpleCNN for CIFAR-10 / 用于 CIFAR-10 的简单 CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# PGD attack / PGD 攻击
def pgd_attack(model, x, y, epsilon, alpha, iters):
    x_adv = x.clone().detach().requires_grad_(True)
    for _ in range(iters):
        model.zero_grad()
        output = model(x_adv)
        loss = F.cross_entropy(output, y)
        loss.backward()
        grad = x_adv.grad.data
        x_adv = x_adv.detach() + alpha * grad.sign()
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
        x_adv = torch.clamp(x_adv, 0, 1).detach().requires_grad_(True)
    return x_adv.detach()


# FGSM attack / FGSM 攻击
def fgsm_attack(model, x, y, epsilon):
    x = x.clone().detach().requires_grad_(True)
    output = model(x)
    loss = F.cross_entropy(output, y)
    model.zero_grad()
    loss.backward()
    x_adv = x + epsilon * x.grad.sign()
    return torch.clamp(x_adv, 0, 1).detach()


# Load CIFAR-10 / 加载数据
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root="./data", train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Train base classifier on clean data / 在干净数据上训练基分类器
def train_classifier(model, loader, epochs, lr=0.001):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[Base Classifier] Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(loader):.4f}")

base_model = SimpleCNN().to(device)
print("Training base classifier...")
train_classifier(base_model, train_loader, epochs=10)

# Evaluate clean accuracy / 干净准确率
def clean_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

print(f"Clean test accuracy: {clean_accuracy(base_model, test_loader):.4f}")


## 解读

基分类器在干净 CIFAR-10 上的准确率约为 65%。检测器不直接使用原始图像，而是利用基分类器的 logit 输出作为特征：对抗样本的 logit 分布通常与干净样本不同，因此可以被二分类器识别。

In [ ]:
# Build detection dataset: logits as features, label 0=clean, 1=adversarial
# 构建检测数据集：以模型 logits 为特征，0=干净样本，1=对抗样本
def build_detection_dataset(model, loader, attack="fgsm", epsilon=0.03, alpha=2/255, iters=10):
    model.eval()
    features = []
    labels = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_clean = x.clone()
        if attack == "fgsm":
            x_adv = fgsm_attack(model, x, y, epsilon)
        elif attack == "pgd":
            x_adv = pgd_attack(model, x, y, epsilon, alpha, iters)
        else:
            raise ValueError("attack must be fgsm or pgd")
        with torch.no_grad():
            logits_clean = model(x_clean)
            logits_adv = model(x_adv)
        features.append(logits_clean.cpu())
        features.append(logits_adv.cpu())
        labels.append(torch.zeros(y.size(0)))
        labels.append(torch.ones(y.size(0)))
    features = torch.cat(features, dim=0)
    labels = torch.cat(labels, dim=0)
    return features, labels

# Generate FGSM dataset for training detector and PGD dataset for generalization test
# 用 FGSM 样本训练检测器，用 PGD 样本测试泛化能力
X_fgsm, y_fgsm = build_detection_dataset(base_model, test_loader, attack="fgsm", epsilon=0.03)
X_pgd, y_pgd = build_detection_dataset(base_model, test_loader, attack="pgd", epsilon=0.03, alpha=2/255, iters=10)

print(f"FGSM dataset shape: {X_fgsm.shape}, labels: {y_fgsm.shape}")
print(f"PGD dataset shape: {X_pgd.shape}, labels: {y_pgd.shape}")

# Split FGSM dataset into train/test / 划分训练集与测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_fgsm, y_fgsm, test_size=0.3, random_state=42, stratify=y_fgsm
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


# Simple detector: MLP on logits / 简单检测器：在 logits 上的 MLP
class Detector(nn.Module):
    def __init__(self):
        super(Detector, self).__init__()
        self.fc1 = nn.Linear(10, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x)).squeeze()


def train_detector(detector, X_train, y_train, epochs=10, lr=0.001, batch_size=64):
    detector.train()
    optimizer = optim.Adam(detector.parameters(), lr=lr)
    criterion = nn.BCELoss()
    n = X_train.size(0)
    for epoch in range(epochs):
        perm = torch.randperm(n)
        total_loss = 0.0
        batches = 0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            bx = X_train[idx].to(device)
            by = y_train[idx].to(device)
            optimizer.zero_grad()
            out = detector(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            batches += 1
        print(f"[Detector] Epoch {epoch + 1}/{epochs}, Loss: {total_loss / batches:.4f}")


detector = Detector().to(device)
print("Training detector on FGSM logits...")
train_detector(detector, X_train, y_train, epochs=10)


In [ ]:
# Evaluate detector / 评估检测器
def evaluate_detector(detector, X, y, threshold=0.5):
    detector.eval()
    with torch.no_grad():
        probs = detector(X.to(device)).cpu().numpy()
    y_pred = (probs >= threshold).astype(int)
    y_true = y.numpy().astype(int)
    return y_true, y_pred

y_true, y_pred = evaluate_detector(detector, X_test, y_test)

print("=== Detection performance on FGSM test set ===")
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")

# Confusion matrix / 混淆矩阵
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix: Clean vs Adversarial (FGSM test set)")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ["Clean", "Adversarial"])
plt.yticks(tick_marks, ["Clean", "Adversarial"])
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, f"{cm[i, j]}", ha="center", va="center",
             color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


## 解读

检测准确率约 67.7% 意味着检测器能正确区分大约三分之二的对抗样本与干净样本。精确率（Precision）和召回率（Recall）进一步说明：如果召回率较高，说明大部分对抗样本不会被漏掉；但精确率不够高也意味着会有一部分干净样本被误报为对抗样本。

In [ ]:
# Generalization: detector trained on FGSM, tested on PGD
# 泛化能力测试：检测器在 FGSM 上训练，在 PGD 上测试
y_true_pgd, y_pred_pgd = evaluate_detector(detector, X_pgd, y_pgd)

print("=== Generalization to PGD attacks (not seen during training) ===")
print(f"Accuracy:  {accuracy_score(y_true_pgd, y_pred_pgd):.4f}")
print(f"Precision: {precision_score(y_true_pgd, y_pred_pgd):.4f}")
print(f"Recall:    {recall_score(y_true_pgd, y_pred_pgd):.4f}")
print(f"F1 Score:  {f1_score(y_true_pgd, y_pred_pgd):.4f}")

cm_pgd = confusion_matrix(y_true_pgd, y_pred_pgd)
plt.figure(figsize=(5, 4))
plt.imshow(cm_pgd, interpolation="nearest", cmap=plt.cm.Oranges)
plt.title("Confusion Matrix: Clean vs Adversarial (PGD generalization)")
plt.colorbar()
plt.xticks(tick_marks, ["Clean", "Adversarial"])
plt.yticks(tick_marks, ["Clean", "Adversarial"])
for i, j in itertools.product(range(cm_pgd.shape[0]), range(cm_pgd.shape[1])):
    plt.text(j, i, f"{cm_pgd[i, j]}", ha="center", va="center",
             color="white" if cm_pgd[i, j] > cm_pgd.max() / 2 else "black")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


## 解读

有趣的是，在 FGSM 上训练的检测器对 PGD 攻击的检测效果反而更好（76.8% vs 67.7%）。这说明 PGD 对抗样本在 logit 空间中的“异常特征”更明显。但这也意味着检测器存在自适应攻击风险：攻击者如果知道检测器的存在，可以优化对抗样本同时绕过检测器，这是检测方法的根本局限。

#### 检测器泛化能力对比

下面比较在 FGSM 样本上训练的检测器，分别在同源攻击（FGSM）和跨攻击（PGD）测试集上的准确率。跨攻击泛化能力通常较弱，这是对抗样本检测器的重要局限。

In [ ]:
# 检测器在同源 vs 跨攻击样本上的准确率对比
y_true_fgsm, y_pred_fgsm = evaluate_detector(detector, X_test, y_test)
acc_fgsm = accuracy_score(y_true_fgsm, y_pred_fgsm)

y_true_pgd, y_pred_pgd = evaluate_detector(detector, X_pgd, y_pgd)
acc_pgd = accuracy_score(y_true_pgd, y_pred_pgd)

categories = ["FGSM 训练\nFGSM 测试", "FGSM 训练\nPGD 测试"]
accuracies = [acc_fgsm, acc_pgd]

plt.figure(figsize=(7, 5))
bars = plt.bar(categories, accuracies, color=["steelblue", "coral"], edgecolor="black")
plt.ylim(0, 1.05)
plt.ylabel("检测准确率")
plt.title("对抗样本检测：同源 vs 跨攻击泛化")
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{acc:.2%}", ha="center", va="bottom", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 小结与讨论

- **检测结果**：在训练分布（FGSM）上，检测器通常能取得较高的准确率、精确率和召回率。
- **泛化挑战**：当测试攻击类型（PGD）与训练时不一致时，检测性能往往会下降，说明检测器对攻击类型敏感。
- **检测的局限性**：对抗样本检测并非万能防御。攻击者可以通过**自适应攻击**（adaptive attack）使扰动绕过已知检测器；因此，检测通常与对抗训练、随机平滑等机制配合使用，形成纵深防御。

检测是一种“事后”防御——它试图在输入进入模型之前拦截对抗样本，与对抗训练（Notebook 27）这种“事前”防御互补。但检测方法面临自适应攻击的挑战：攻击者知道检测器的存在后，可以优化对抗样本同时绕过检测器。
